# 6. Longitudinal Analysis

In this notebook, we examine how the infant gut microbiome changes over time by following children across multiple sampling points. We begin by preparing the data for longitudinal analysis, which requires creating a properly structured dataframe where each sample is linked to its subject ID, age, covariates, and the selected diversity metric. After this, we generate an additional dataframe that captures how much the microbiome changes between visits. This is done using first differences, which measure how an alpha-diversity value changes from one timepoint to the next. These derived data help us quantify both the direction and the magnitude of microbiome change over time.

Using this prepared data, we then apply several longitudinal tools. Volatility plots show how diversity (or its changes) develops over time for each child, giving a clear picture of individual trajectories. Feature volatility identifies which bacterial taxa change the most with age, helping us understand which microbes are important during gut microbiome maturation. Finally, Linear Mixed-Effects Models (LME) allow us to test whether factors such as age, diet, or geography have a significant influence on microbiome development while accounting for repeated measurements from the same child.

### Notebook Structure

**1.** Data preparation  
**2.** Volatility plots  
**3.** Feature volatility  
**4.** Linear Mixed-Effects Models (LME)

### Import Packages

In [1]:
# Import all necessary packages
# Import all necessary packages
import os
import IPython
import pandas as pd
import matplotlib.pyplot as plt
import qiime2 as q2
from qiime2 import Visualization
from ipywidgets import Dropdown, VBox

%matplotlib inline

### Set Working Directory
Ensure that the working directory is correctly set to the 'scripts' folder within the main project directory. 
Otherwise, the file paths used in this notebook may not work properly.

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# 3 - Data directories
raw_data_dir = "../data/raw"
processed_data_dir = "../data/processed"

meta_data_dir = "../data/processed/metadata"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
phylogeny_data_dir = "../data/processed/phylogeny"
diversity_data_dir = "../data/processed/diversity"
longitudinal_data_dir = "../data/processed/longitudinal"

# Create directories
!mkdir -p $raw_data_dir $processed_data_dir $meta_data_dir \
         $denoising_data_dir $taxonomy_data_dir $phylogeny_data_dir \
         $diversity_data_dir $longitudinal_data_dir

In [4]:
%%bash -s "$longitudinal_data_dir"
mkdir -p "$1"

### Set run and metric to work with

This menu lets you select the run folder you want to analyze, as well as the alpha-diversity metric and the time variable.
The run selector lists all processed datasets available in ../data/processed.
You can choose either Faith’s PD or Shannon as the alpha metric.
For longitudinal analysis, you can select whether to use ages in months or days.
These settings control how the following analysis cells load and process the data.

In [5]:
# List available run directories
run_options = sorted([
    d for d in os.listdir(diversity_data_dir)
    if os.path.isdir(os.path.join(diversity_data_dir, d))
])

run_selector = Dropdown(
    options=run_options,
    description="Run:",
)

# Options for alpha-diversity metrics
alpha_selector = Dropdown(
    options=["faith_pd", "shannon"],
    value="faith_pd",
    description="Alpha metric:",
)

# Options for time variables
time_selector = Dropdown(
    options=["age_months", "age_days"],
    value="age_months",
    description="Time column:",
)

VBox([run_selector, alpha_selector, time_selector])

In [6]:
run = run_selector.value
alpha_metric = alpha_selector.value
time_column = time_selector.value

## 1. Data Preparation  

### 1.1 Export and merge raw alpha diversity

Here we export the alpha diversity QZA file, convert it to a TSV file,
and merge it with the sample metadata.

This gives us one metadata table that includes:

- the sample ID  
- all sample-level metadata  
- the alpha diversity value for that sample  

Many QIIME longitudinal tools require this combined metadata table.

In [7]:
# New combined folder
analysis_dir = f"{longitudinal_data_dir}/{run}/analysis_{alpha_metric}_{time_column}"

# Create base analysis folder + subfolders
alpha_raw_dir        = f"{analysis_dir}/alpha_raw"
alpha_fd_dir         = f"{analysis_dir}/alpha_firstdiff"
volatility_raw_dir   = f"{analysis_dir}/volatility_raw"
volatility_fd_dir    = f"{analysis_dir}/volatility_firstdiff"

!mkdir -p {alpha_raw_dir} {alpha_fd_dir} {volatility_raw_dir} {volatility_fd_dir}

analysis_dir

'../data/processed/longitudinal/None/analysis_faith_pd_age_months'

In [8]:
# Export + merge raw alpha metric

alpha_qza = f"{diversity_data_dir}/{run}/alpha/metrics/{alpha_metric}_vector.qza"
alpha_export = f"{alpha_raw_dir}/export"

!qiime tools export --input-path {alpha_qza} --output-path {alpha_export}

df_alpha = pd.read_csv(f"{alpha_export}/alpha-diversity.tsv", sep="\t")
df_alpha = df_alpha.rename(columns={"#SampleID": "id", "Unnamed: 0": "id"})

metadata = pd.read_csv(f"{meta_data_dir}/metadata_merged.tsv", sep="\t")

merged_alpha = metadata.merge(df_alpha, on="id", how="inner")

alpha_out = f"{alpha_raw_dir}/metadata_raw.tsv"
merged_alpha.to_csv(alpha_out, sep="\t", index=False)

alpha_out

Usage: qiime tools export [OPTIONS]

  Exporting extracts (and optionally transforms) data stored inside an
  Artifact or Visualization. Note that Visualizations cannot be transformed
  with --output-format

Options:
  --input-path ARTIFACT/VISUALIZATION
                        Path to file that should be exported        [required]
  --output-path PATH    Path to file or directory where data should be
                        exported to                                 [required]
  --output-format TEXT  Format which the data should be exported as. This
                        option cannot be used with Visualizations
  --help                Show this message and exit.

                    There was a problem with the command:                     
 (1/1) Invalid value for '--input-path': File
  '../data/processed/diversity/None/alpha/metrics/faith_pd_vector.qza' does
  not exist.


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/longitudinal/None/analysis_faith_pd_age_months/alpha_raw/export/alpha-diversity.tsv'

### 1.2 Compute and merge first differences of alpha diversity 

First differences measure how much alpha diversity changes between
two consecutive visits:

``difference = alpha_(t2) – alpha_(t1)``

This tells us how dynamic or stable the microbiome is between sampling moments.

In [ ]:
# Compute first differences

fd_qza = f"{alpha_fd_dir}/{alpha_metric}_firstdiff.qza"

!qiime longitudinal first-differences \
  --p-metric {alpha_metric} \
  --m-metadata-file {alpha_out} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --p-replicate-handling random \
  --o-first-differences {fd_qza} \
  --verbose

We export the first-difference QZA file and merge it with the metadata.

This creates a second metadata table specifically for analyzing
the *change* in diversity instead of the raw values.

In [ ]:
# Export + merge first differences

fd_export = f"{alpha_fd_dir}/export"
!qiime tools export --input-path {fd_qza} --output-path {fd_export}

df_fd = pd.read_csv(f"{fd_export}/FirstDifferences.tsv", sep="\t")
df_fd = df_fd.rename(columns={"#SampleID": "id", "Unnamed: 0": "id"})

merged_fd = metadata.merge(df_fd, on="id", how="inner")

fd_out = f"{alpha_fd_dir}/metadata_firstdiff.tsv"
merged_fd.to_csv(fd_out, sep="\t", index=False)

fd_out

## 2. Volatility Analysis

A volatility plot shows how a metric, such as Shannon diversity, changes over time for each child in the study. It is a useful first step because it gives an immediate visual impression of how the microbiome develops across age. By looking at the lines for all children together, we can see whether diversity tends to increase, decrease, or stay stable as children grow. It also shows how similar or different the developmental patterns are between children. Some children might have very smooth increases in diversity, while others may show more fluctuations. This helps us understand the structure of the data before doing any statistical modeling. If the lines mostly slope upward, it suggests that diversity increases with age, which is what we often expect in early-life microbiome development. The plot also helps us judge whether a linear mixed-effects model makes sense, because we can visually check whether there appears to be a consistent trend that could be captured by a statistical model.

### 2.1 Alpha metric (raw)

In this part we focus on the raw alpha diversity values. Later we also look at first differences, which represent how much the diversity changes between two consecutive timepoints. That helps us study not only the level of diversity but also how quickly the microbiome is changing over time.

In [ ]:
# Volatility on raw alpha metric

vol_raw_out = f"{volatility_raw_dir}/volatility_raw.qzv"

!qiime longitudinal volatility \
  --m-metadata-file {alpha_out} \
  --p-default-metric {alpha_metric} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --o-visualization {vol_raw_out}

In [ ]:
Visualization.load(vol_raw_out)

### 2.2 Alpha metric (first differences)

In [ ]:
#  Volatility on first differences

vol_fd_out = f"{volatility_fd_dir}/volatility_firstdiff.qzv"

!qiime longitudinal volatility \
  --m-metadata-file {fd_out} \
  --p-default-metric Difference \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --o-visualization {vol_fd_out}

In [ ]:
Visualization.load(vol_fd_out)

## 3. Feature Volatility

In this section, we investigate which microbial features (ASVs or taxa) change the most with age.  
Feature volatility uses a machine learning model (based on random forests) to determine how strongly the microbiome reflects
temporal development across infancy.

In [ ]:
## Takes some time, quick coffee break!

cmd = f"""
qiime longitudinal feature-volatility \
  --i-table {diversity_data_dir}/{run}/rarefied_table.qza \
  --m-metadata-file {alpha_out} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --output-dir {analysis_dir}/feature_volatility_raw \
  --verbose
"""

print(cmd)
!{cmd}

The next visualization shows how the most important microbial features change across the age range.
Each curve represents one taxon, and its shape indicates how its abundance shifts over time.

In [ ]:
Visualization.load(f"{analysis_dir}/feature_volatility_raw/volatility_plot.qzv")

The next visualization summarizes how well the Random Forest regressor predicts infant age.
It includes:

- Correlation between predicted and actual age  
- R² value (variance explained)  
- Mean squared error (MSE)  
- Slope and intercept of the regression line  

Together, these indicate how strongly the microbiome reflects developmental age.

In [ ]:
Visualization.load(f"{analysis_dir}/feature_volatility_raw/accuracy_results.qzv")

In this step, we take the feature importance values from the feature-volatility analysis and match
them with their taxonomy. The Random Forest model tells us which microbial features (ASVs) are most
helpful for predicting a child’s age, but the output only shows ASV IDs, which aren’t meaningful on
their own. By merging these IDs with the taxonomy file, we can finally see what these features
actually are — for example, which phylum, family, or genus they belong to.

This is important because it lets us interpret the model in a biological way. Instead of just
knowing that “Feature_123” is important, we can see whether it’s a Bacteroides, a Blautia, or
something else that changes with age. After merging, we apply a small cutoff (0.02 by default) to
focus only on the ASVs that the model considers genuinely important. The final printed list shows
the feature ID, its importance score, and its full taxonomy, making it easier to understand which
microbes are actually driving age-related patterns in the dataset.


In [ ]:
# Export feature importance from the raw feature-volatility output
!qiime tools export \
  --input-path {analysis_dir}/feature_volatility_raw/feature_importance.qza \
  --output-path {analysis_dir}/feature_volatility_raw/importance_export

# Export taxonomy so we can annotate the features
!qiime tools export \
  --input-path {taxonomy_data_dir}/taxonomy.qza \
  --output-path {taxonomy_data_dir}/taxonomy_export

# Load importance values + taxonomy
importance_path = f"{analysis_dir}/feature_volatility_raw/importance_export/importance.tsv"
taxonomy_path = f"{taxonomy_data_dir}/taxonomy_export/taxonomy.tsv"

df_imp = pd.read_csv(importance_path, sep="\t")
df_tax = pd.read_csv(taxonomy_path, sep="\t")

# Merge importance + taxonomy
merged = df_imp.merge(df_tax, left_on="feature", right_on="Feature ID", how="left")

# Filter by importance threshold
threshold = 0.02  # adjust as needed
top_features = (
    merged[merged["importance"] >= threshold]
    .sort_values("importance", ascending=False)
)

# Print top features with taxonomy
for _, row in top_features.iterrows():
    print("Feature ID:", row["feature"])
    print("Importance:", round(row["importance"], 4))
    print("Taxonomy: ", row["Taxon"] if "Taxon" in row else row.get("taxonomy", "N/A"))
    print("-" * 80)

These feature-volatility results are interesting because they point out which microbes change the most as children grow older and therefore which ones help the model predict age. The top feature in the list belongs to the genus Blautia. Blautia is part of the Lachnospiraceae family and is generally more common in older children and adults than in young infants. Its high importance score suggests that its abundance increases in a steady, predictable way as infants age. This makes it a strong signal for the model and shows that Blautia may be one of the microbes that mark the transition from an infant-type microbiome to a more mature one.

The next important microbe is Thomasclavelia, which is part of the Erysipelotrichaceae family. This group often appears in studies of early-life microbiome development because it tends to shift during infancy when the gut is still forming its structure. Its high importance score means that the model is detecting a clear pattern in how this microbe changes with age, which suggests it also plays a role in the normal progression of gut maturation.

Flavonifractor and Faecalibacterium also show up in the list. These microbes are usually linked to a more stable and mature gut ecosystem. Their appearance among the important features suggests that they become more common in slightly older infants. This fits with what we know about gut development, because these taxa are often associated with the shift toward an adult-like microbiome.

Overall, these results are interesting because the model highlights microbes that match our expectations of how the infant gut develops. The taxa with high importance scores are the ones that are known to change as infants grow, and seeing them appear here suggests that the feature-volatility method is capturing real biological patterns rather than random variation.

## 4. Linear Mixed-Effects Model

In this step we use a linear mixed-effects model to examine how a specific metadata variable, such as geo_location_name, is associated with the raw alpha diversity values over time. This type of model is appropriate for longitudinal data because each child has multiple samples, and these repeated measurements are not independent from each other. By using an LME model we can take into account that samples from the same child belong together, while still testing whether age or another variable has an influence on the diversity values.

The goal here is to understand whether the alpha diversity follows different patterns depending on the group we are testing, and whether these patterns are statistically meaningful. We only apply this analysis to the raw alpha metric because LME models are designed to look at how the level of a metric changes over time rather than the short-term fluctuations captured by first differences. The output is a QIIME2 visualization that reports the estimated effects, significance values, and model interpretation, helping us see whether the tested variable plays a role in shaping microbiome development.


In [ ]:
# Choose which metadata column you want to test
group_to_test = "geo_location_name"   # other metrics include geo_location_name and treatment_exposure 

# Create output directory
lme_dir = f"{analysis_dir}/linear_mixed_effects_raw_{group_to_test}"
!mkdir -p {lme_dir}

# Path to the raw alpha metadata table
raw_metadata = f"{analysis_dir}/alpha_raw/metadata_raw.tsv"
print("Using metadata:", raw_metadata)

# Run LME
cmd_lme = f"""
qiime longitudinal linear-mixed-effects \
  --m-metadata-file {raw_metadata} \
  --p-metric {alpha_metric} \
  --p-state-column {time_column} \
  --p-individual-id-column host_id \
  --p-group-columns {group_to_test} \
  --o-visualization {lme_dir}/lme-{alpha_metric}-{group_to_test}.qzv
"""

print(cmd_lme)
!{cmd_lme}

In [ ]:
Visualization.load(f"{lme_dir}/lme-{alpha_metric}-{group_to_test}.qzv")

# References




Chen Hong, Microbiome Analysis: From Raw Reads to Ecological Interpretation, Frontiers in Microbiology, 2022. https://pmc.ncbi.nlm.nih.gov/articles/PMC9285460/

National Cancer Institute, Center for Cancer Research. QIIME 2: Lesson 5 — Longitudinal Data Analysis. https://bioinformatics.ccr.cancer.gov/docs/qiime2/Lesson5/

Amy D. Willis, Emily R. K. Martin, Estimating Diversity in Networked Ecological Communities, mSystems, 2018, Volume 3, Issue 3. https://doi.org/10.1128/msystems.00219-18

QIIME 2 Documentation. (2022). Longitudinal Analysis Tutorial — Linear Mixed Effects Models. https://docs.qiime2.org/2022.8/tutorials/longitudinal/#linear-mixed-effect-models